# Introduction - Employment data by State as of May 2025
This notebook outlines the process of transforming data and using it to create an interactable dashboard for State employment and wage statistics.

The primary purpose of this notebook is to visualize differences in hourly pay between major occupational groups across the U.S. There will be Nationwide and Statewide comparisons. Hourly means, and varying percentiles will be used. The median earnings across States will be used as a baseline for comparison.

Questions to answer:

1. What states pay the highest and lowest median wages?

2. What occupational group do the highest median earners come from?

3. How does Michigan compare to the national median wages per occupational group?

4. Which States have the highest proportion of Computer and Mathematical occupations?

# Library choice and fit
If you haven't installed these modules, run the below cell to do so.

In [1]:
%pip install panel pandas openpyxl plotly "nbformat>=4.2.0" jupyter_bokeh

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'C:\Users\strid\AppData\Local\Programs\MUEDIT~1\Python\pythonw.exe -m pip install --upgrade pip' command.



I chose plotly as the primary library for this exploration. I was originally planning on going with seaborn, due to it being described as a more modern, feature rich, version of matplotlib, however, it did not have the mapping functionality that I was hoping to use. It is declarative.

Created by: Plotly Technologies Inc.

License: MIT License


In [2]:
# Import relevant libraries
import pandas as pd 
import panel as pn
import plotly.express as px
import plotly.graph_objects as go
import jupyter_bokeh
from pathlib import Path

# Data - Cleaning and Transformations

Source: https://www.bls.gov/oes/tables.htm -> State (XLSX)

License Type: Open License (see https://resources.data.gov/open-licenses/)

Size: 7496 KB, 32 Columns, 37409 Rows

Field references/Columns
(Only including relevant fields used in this notebook. For full descriptions, see raw data. * Some columns used for filtering aren't included)
| Field | Field Description |
| --- | --- |
| `area` | U.S. (99), state FIPS code, Metropolitan Statistical Area (MSA) code, or OEWS-specific nonmetropolitan area code. |
| `area_title` | Area name. |
| `area_type` | Area type: 1 = U.S.; 2 = State; 3 = U.S. Territory; 4 = Metropolitan Statistical Area (MSA); 6 = Nonmetropolitan Area. |
| `tot_emp` | Estimated total employment rounded to the nearest 10 (excludes self-employed). |
| `emp_prse` | Percent relative standard error (PRSE) for the employment estimate. PRSE is a measure of model error, expressed as a percentage of the corresponding estimate. Model error arises when inferences about a population are made from a survey of selected members of that population, instead of data from all members of the population. Estimates with lower PRSEs are typically more precise in the presence of model error. |
| `jobs_1000` | The number of jobs (employment) in the given occupation per 1,000 jobs in the given area. Only available for the state and MSA estimates; otherwise, this column is blank. |
| `pct_total` | Percent of industry employment in the given occupation. Percents may not sum to 100 because the totals may include data for occupations that could not be published separately. Only available for the national industry estimates; otherwise, this column is blank. |
| `pct_rpt` | Percent of establishments reporting the given occupation for the cell. Only available for the national industry estimates; otherwise, this column is blank. |
| `h_mean` | Mean hourly wage. |
| `mean_prse` | Percent relative standard error (PRSE) for the mean wage estimate. PRSE is a measure of model error, expressed as a percentage of the corresponding estimate. Model error arises when inferences about a population are made from a survey of selected members of that population, instead of data from all members of the population. Estimates with lower PRSEs are typically more precise in the presence of model error. |
| `h_pct10` | Hourly 10th percentile wage. |
| `h_pct25` | Hourly 25th percentile wage. |
| `h_median` | Hourly median wage (or the 50th percentile). |
| `h_pct75` | Hourly 75th percentile wage. |
| `h_pct90` | Hourly 90th percentile wage. |


Notes:	

"*"  = indicates that a wage estimate is not available	

** = indicates that an employment estimate is not available	

"#" = indicates a wage equal to or greater than $115.00 per hour or $239,200 per year 	

~ = indicates that the percent of establishments reporting the occupation is less than 0.5%	

In [3]:
# Open the Excel file and read the data into a pandas DataFrame
df = pd.read_excel(r"C:\Users\strid\OneDrive\Desktop\MADS 521\Raw_Data\state_M2025_dl.xlsx")
df


,AREA,AREA_TITLE,AREA_TYPE,PRIM_STATE,NAICS,NAICS_TITLE,I_GROUP,OWN_CODE,OCC_CODE,OCC_TITLE,...,H_MEDIAN,H_PCT75,H_PCT90,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY
0,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,00-0000,All Occupations,...,21.96,32.11,48.68,26090,32050,45670,66790,101250,NaN,NaN
1,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,49.99,72.94,100.05,56920,77180,103980,151720,208110,NaN,NaN
2,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,80.01,106.96,152.6,101670,134650,166420,222490,317400,NaN,NaN
3,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-1021,General and Operations Managers,...,53.75,81.5,117.11,55160,76840,111790,169510,243600,NaN,NaN
4,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-2011,Advertising and Promotions Managers,...,31.08,68.85,102.82,60000,62500,64640,143210,213860,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37403,78,Virgin Islands,3,VI,0,Cross-industry,cross-industry,1235,53-7051,Industrial Truck and Tractor Operators,...,16.94,20.88,25.29,27190,31030,35230,43420,52600,NaN,NaN
37404,78,Virgin Islands,3,VI,0,Cross-industry,cross-industry,1235,53-7061,Cleaners of Vehicles and Equipment,...,13.6,14.99,17.6,22230,26360,28290,31190,36610,NaN,NaN
37405,78,Virgin Islands,3,VI,0,Cross-industry,cross-industry,1235,53-7062,"Laborers and Freight, Stock, and Material Move...",...,15.3,17.99,18.72,26620,28150,31810,37410,38940,NaN,NaN
37406,78,Virgin Islands,3,VI,0,Cross-industry,cross-industry,1235,53-7064,"Packers and Packagers, Hand",...,11,13.08,17.07,21890,21960,22880,27200,35510,NaN,NaN


*Please note that Percent relative standard error (PRSE) differs. Please refer to raw data. Only data with PRSE scores under 25% are kept/used for these visuals.

In [4]:
# Include only State data (DC included)
df = df[df["AREA_TYPE"] == 2]

# Preserve all-occupations estimates before removing them.
all_occupations = df.loc[
    df["OCC_TITLE"] == "All Occupations"
].copy()

# Remove the "All Occupations" rows so the totals aren't include with the main data
df = df[df["OCC_TITLE"] != "All Occupations"]

# Only include rows with major occupation types in the main DataFrame
df = df[df["O_GROUP"] == "major"]

# Keep values where the PRSE is known and below 25%. Remove non-numeric hourly values as well.
numeric_cols = ["TOT_EMP","JOBS_1000","H_MEAN","H_MEDIAN","H_PCT10","H_PCT25","H_PCT75","H_PCT90","EMP_PRSE"]

# Make a separate DataFrame for 
all_occupations[numeric_cols] = (
    all_occupations[numeric_cols]
    .apply(pd.to_numeric, errors="coerce")
)

all_occupations = all_occupations.loc[
    all_occupations["EMP_PRSE"].lt(25)
    & all_occupations["H_MEDIAN"].notna()
].copy()


df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric,errors="coerce")
df = df.dropna(subset=["EMP_PRSE","H_MEDIAN"])
df = df[df["EMP_PRSE"] < 25]


# Only include rows with viable hourly data
df["H_MEAN"] = df["H_MEAN"].dropna()



# Keep only relevant columns for hourly analysis
df = df[["AREA_TITLE","PRIM_STATE","OCC_TITLE","TOT_EMP","JOBS_1000","H_MEAN","H_MEDIAN","H_PCT10","H_PCT25","H_PCT75","H_PCT90"]]

# Remove unnecessary "Occupations" after each occupation title. This can make some titles grammatically incorrect, but it's worth it for the visual clarity.
df["OCC_TITLE"] = df["OCC_TITLE"].str.replace(" Occupations","")

df = df.reset_index(drop=True)





In [5]:
df.head()

,AREA_TITLE,PRIM_STATE,OCC_TITLE,TOT_EMP,JOBS_1000,H_MEAN,H_MEDIAN,H_PCT10,H_PCT25,H_PCT75,H_PCT90
0,Alabama,AL,Management,112960,53.443,58.94,49.99,27.37,37.11,72.94,100.05
1,Alabama,AL,Business and Financial Operations,108150,51.169,40.68,36.73,21.83,27.74,49.03,64.12
2,Alabama,AL,Computer and Mathematical,57370,27.142,49.97,48.09,24.16,33.19,63.40,79.60
3,Alabama,AL,Architecture and Engineering,49580,23.458,51.69,48.07,26.45,36.65,64.02,82.34
4,Alabama,AL,"Life, Physical, and Social Science",14570,6.894,39.13,35.61,21.79,26.73,46.98,61.55


With the data cleaned, I'll make three separate groupings. One by occupation, one by state, and one by State & occupation.



In [6]:
# Used for state totals
df_by_state = df.pivot_table(values = ["JOBS_1000","H_MEAN","H_MEDIAN","H_PCT10","H_PCT25","H_PCT75","H_PCT90"],
                                  index = ["PRIM_STATE"]).round(2)

df_by_state_and_oc = df.pivot_table(values = ["JOBS_1000","H_MEAN","H_MEDIAN","H_PCT10","H_PCT25","H_PCT75","H_PCT90"],
                                  index = ["PRIM_STATE","OCC_TITLE"])

# Used for occupations
df_by_occupation = df.pivot_table(values = ["JOBS_1000","H_MEAN","H_MEDIAN","H_PCT10","H_PCT25","H_PCT75","H_PCT90"],
                                  index = ["OCC_TITLE"]).round(2)

# Used for unique All Occupation logic
summary_columns = [
    "JOBS_1000",
    "H_MEDIAN",
    "H_PCT10",
    "H_PCT25",
    "H_PCT75",
    "H_PCT90",
]

if all_occupations["PRIM_STATE"].duplicated().any():
    raise ValueError(
        "Multiple All Occupations rows exist for a state. "
        "Check industry and ownership scope before proceeding."
    )

all_occupations_summary = (
    all_occupations
    .set_index(["PRIM_STATE", "OCC_TITLE"])[summary_columns]
)

df_by_state_and_oc = pd.concat([
    df_by_state_and_oc,
    all_occupations_summary,
]).sort_index()

# Average the actual state-level All Occupations estimates.
# This matches the across-state averaging used for other groups.
all_occupations_benchmark = (
    all_occupations_summary[summary_columns]
    .mean()
    .to_frame()
    .T
    .round(2)
)

all_occupations_benchmark.index = pd.Index(
    ["All Occupations"],
    name="OCC_TITLE",
)

# Replace an existing row if this cell is rerun.
df_by_occupation = pd.concat([
    df_by_occupation.drop(
        index="All Occupations",
        errors="ignore",
    ),
    all_occupations_benchmark,
]).sort_index()

In [7]:
df_by_state.head()

,H_MEAN,H_MEDIAN,H_PCT10,H_PCT25,H_PCT75,H_PCT90,JOBS_1000
PRIM_STATE,,,,,,,
AK,36.79,33.26,20.65,25.40,43.14,55.66,45.45
AL,29.90,26.60,15.97,20.07,35.72,47.49,45.45
AR,27.89,24.25,15.40,18.88,32.14,42.98,45.45
AZ,33.36,29.36,19.10,22.97,38.78,51.06,45.45
CA,41.52,35.97,21.78,26.65,50.12,66.69,45.45


In [8]:
df_by_state_and_oc.head()

H_MEAN  H_MEDIAN  \
PRIM_STATE OCC_TITLE                                                          
AK         All Occupations                                    NaN     29.33   
           Architecture and Engineering                     55.59     51.12   
           Arts, Design, Entertainment, Sports, and Media   30.42     25.86   
           Building and Grounds Cleaning and Maintenance    20.91     19.48   
           Business and Financial Operations                43.60     40.01   

                                                           H_PCT10  H_PCT25  \
PRIM_STATE OCC_TITLE                                                          
AK         All Occupations                                   16.79    20.90   
           Architecture and Engineering                      32.02    39.73   
           Arts, Design, Entertainment, Sports, and Media    17.85    20.69   
           Building and Grounds Cleaning and Maintenance     14.40    17.41   
           Business and Financial Operations                 24.69    31.44   

                                                           H_PCT75  H_PCT90  \
PRIM_STATE OCC_TITLE                                                          
AK         All Occupations                                   44.67    61.89   
           Architecture and Engineering                      65.42    83.81   
           Arts, Design, Entertainment, Sports, and Media    36.73    49.78   
           Building and Grounds Cleaning and Maintenance     23.38    28.16   
           Business and Financial Operations                 52.24    63.81   

                                                           JOBS_1000  
PRIM_STATE OCC_TITLE                                                  
AK         All Occupations                                  1000.000  
           Architecture and Engineering                       19.931  
           Arts, Design, Entertainment, Sports, and Media      9.819  
           Building and Grounds Cleaning and Maintenance      29.677  
           Business and Financial Operations                  45.421

In [9]:
df_by_occupation.head()

,H_MEAN,H_MEDIAN,H_PCT10,H_PCT25,H_PCT75,H_PCT90,JOBS_1000
OCC_TITLE,,,,,,,
All Occupations,NaN,25.13,14.89,18.25,38.50,56.97,1000.00
Architecture and Engineering,49.31,46.11,27.64,35.72,60.14,74.76,17.08
"Arts, Design, Entertainment, Sports, and Media",33.38,27.80,15.99,20.33,39.34,53.97,12.53
Building and Grounds Cleaning and Maintenance,19.32,18.20,14.25,15.81,21.41,25.60,31.04
Business and Financial Operations,43.34,39.21,23.71,30.06,51.71,66.88,64.39


# Visualization techniques
With the data ready to be used, the next step is to create the 4 charts to be used on the dashboard.

The first chart is a dotplot that compares the median earnings of each occupational group for two different States. It's useful for comparing earnings by State, and answering question 3.

It has widgets for the two States being compared (PRIM_STATE), and the summary stat being used. It uses the df_by_state DataFrame.

In [10]:

def make_state_comparison(state_1, state_2, statistic="H_MEDIAN"):
    """Compare two states using a dot plot of occupational summaries."""

    if state_1 == state_2:
        raise ValueError("Select two different states.")

    # Both state and occupation are required for this chart.
    data = df_by_state_and_oc.copy()


    data = data.reset_index()
    data = data.loc[
        data["OCC_TITLE"] != "All Occupations"
    ].copy()
    # Summary Statistic = all non-grouped columns except JOBS_1000.
    valid_statistics = [
        col for col in data.columns
        if col not in ["PRIM_STATE", "OCC_TITLE", "JOBS_1000"]
    ]

    if statistic not in valid_statistics:
        raise ValueError(
            f"Choose a Summary Statistic from: {valid_statistics}"
        )

    data[statistic] = pd.to_numeric(
        data[statistic], errors="coerce"
    )

    selected = data.loc[
        data["PRIM_STATE"].isin([state_1, state_2]),
        ["PRIM_STATE", "OCC_TITLE", statistic],
    ].dropna(subset=["OCC_TITLE", statistic])

    # Keep occupations with valid values in BOTH states.
    paired = selected.pivot(
        index="OCC_TITLE",
        columns="PRIM_STATE",
        values=statistic,
    )

    paired = paired.reindex(
        columns=[state_1, state_2]
    ).dropna(subset=[state_1, state_2])

    # Sort occupations by their average value across the two states.
    occupation_order = (
        paired.mean(axis=1)
        .sort_values(ascending=False)
        .index.tolist()
    )

    # Return to long format for Plotly.
    plot_data = (
        paired.reset_index()
        .melt(
            id_vars="OCC_TITLE",
            value_vars=[state_1, state_2],
            var_name="PRIM_STATE",
            value_name="VALUE",
        )
    )

    units = " ($/hour)" if statistic.startswith("H_") else ""

    fig = px.scatter(
        plot_data,
        x="VALUE",
        y="OCC_TITLE",
        color="PRIM_STATE",
        symbol="PRIM_STATE",
        color_discrete_map={
            state_1: "#0072B2",
            state_2: "#D55E00",
        },
        symbol_map={
            state_1: "circle",
            state_2: "square",
        },
        category_orders={
            "PRIM_STATE": [state_1, state_2],
            "OCC_TITLE": occupation_order,
        },
        labels={
            "VALUE": f"{statistic}{units}",
            "OCC_TITLE": "Occupational group",
            "PRIM_STATE": "State",
        },
        title=f"{statistic} by occupational group: {state_1} versus {state_2}",
    )

    fig.update_traces(
        marker=dict(size=11, opacity=0.85),
    )

    fig.update_layout(
        template="plotly_white",
        height=max(450, len(occupation_order) * 30 + 150),
        margin=dict(l=30, r=30, t=80, b=60),
        legend_title_text="State",
    )

    fig.update_xaxes(showgrid=True)
    fig.update_yaxes(showgrid=True, automargin=True)

    if paired.empty:
        fig.add_annotation(
            text="No occupational groups have valid values in both states.",
            x=0.5,
            y=0.5,
            xref="paper",
            yref="paper",
            showarrow=False,
        )

    return fig
# Preview
fig = make_state_comparison("MI", "CA", "H_MEDIAN")
fig

The second chart is a boxplot of national averages for all occupations. This chart helps to give an understanding of the wage distributions for multiple occupational groups, and answering question 2. I wanted to use a box/whisker plot despite not having the max and mins, as it still seemed to be the best method for visually comparing the summary stats of multiple occupations


It has a dropdown widget for the occupational group being compared (up to 5 at once). It uses the df_by_occupation DataFrame. ***Whiskers included are for the 10th and 90th percentiles, which is non-standard. Due to only having summary statistics, a standard plot could not be created.

In [11]:
def make_occupation_boxplot(selected_occupations):
    """Create percentile boxes without whiskers for up to five groups."""

    if selected_occupations is None:
        selected_occupations = []

    # Accept either one occupation string or a list.
    if isinstance(selected_occupations, str):
        selected_occupations = [selected_occupations]

    # Remove duplicate selections while preserving their order.
    selected_occupations = list(
        dict.fromkeys(selected_occupations)
    )

    if len(selected_occupations) > 5:
        raise ValueError(
            "Select no more than five occupational groups."
        )

    data = df_by_occupation.copy()


    data = data.reset_index()

    percentile_columns = [
    "H_PCT10",
    "H_PCT25",
    "H_MEDIAN",
    "H_PCT75",
    "H_PCT90",
    ]

    data[percentile_columns] = data[percentile_columns].apply(
        pd.to_numeric,
        errors="coerce",
    )

    available = set(data["OCC_TITLE"].dropna())

    unknown = [
        occupation
        for occupation in selected_occupations
        if occupation not in available
    ]

    if unknown:
        raise ValueError(
            f"Unknown occupational groups: {unknown}"
        )

    selected = (
        data.set_index("OCC_TITLE")
        .reindex(selected_occupations)
        .dropna(subset=percentile_columns)
    )

    # Check that the median falls inside the box.
    ordered = (
        (selected["H_PCT10"] <= selected["H_PCT25"])
        & (selected["H_PCT25"] <= selected["H_MEDIAN"])
        & (selected["H_MEDIAN"] <= selected["H_PCT75"])
        & (selected["H_PCT75"] <= selected["H_PCT90"])
    )

    if not ordered.all():
        invalid_groups = selected.index[~ordered].tolist()
        raise ValueError(
            f"Percentiles are out of order for: {invalid_groups}"
        )

    fig = go.Figure()

    for occupation, row in selected.iterrows():
        fig.add_trace(
            go.Box(
                x=[occupation],
                name=occupation,
                q1=[row["H_PCT25"]],
                median=[row["H_MEDIAN"]],
                q3=[row["H_PCT75"]],

                # Whiskers at the 10th and 90th percentiles.
                lowerfence=[row["H_PCT10"]],
                upperfence=[row["H_PCT90"]],
                whiskerwidth=0.5,

                boxpoints=False,
                showlegend=False,
                fillcolor="rgba(0, 114, 178, 0.35)",
                line=dict(color="#0072B2", width=2),
            )
        )

    fig.update_layout(
        title=dict(
            text=(
                "Occupational wage percentile summaries"
                "<br><sup>"
                "Box: 25th–75th percentiles | Middle line: median | "
                "**Non-Standard Whiskers: 10th and 90th percentiles"
                "</sup>"
            )
        ),
        xaxis_title="Occupational group",
        yaxis_title="Hourly wage summary ($/hour)",
        boxmode="group",
        template="plotly_white",
        height=600,
        margin=dict(l=60, r=30, t=110, b=160),
    )

    fig.update_xaxes(
        type="category",
        categoryorder="array",
        categoryarray=selected_occupations,
        tickangle=-25,
        automargin=True,
    )

    fig.update_yaxes(
        showgrid=True,
        rangemode="tozero",
    )

    if selected.empty:
        fig.add_annotation(
            text="Select up to five occupational groups with valid data.",
            x=0.5,
            y=0.5,
            xref="paper",
            yref="paper",
            showarrow=False,
        )

    return fig

fig = make_occupation_boxplot([
    "Management",
    "Legal",

])

fig

The third chart is a choropleth map of the U.S. with all States. It has the option of choosing one summary statistic, and the ability to compare multiple occupational groups. This chart gives a visual of the nation as a whole, which allows for ease of moving between many States in a short time, and is useful for answering question 1. In addition, this chart can compare the proportion of jobs per 1000 by State. It is useful for finding states with a higher proportion of a certain occupational group/industry, and may be relevant for finding where to look for a job. It also is useful for answering question 4.

It has a dropdown widget for the occupational group being compared, and the summary stat being used. It also has a widget for if it's comparing the proportion of jobs by state. It uses the df_by_state_and_oc DataFrame.

In [12]:

def make_occupation_choropleth(
    occupation,
    statistic="H_MEDIAN",
    map_measure="Summary Statistic",
):
    """Map a selected wage statistic or jobs per 1,000 by state."""

    data = df_by_state_and_oc.reset_index().copy()

    valid_statistics = [
        column
        for column in data.columns
        if column not in ["PRIM_STATE", "OCC_TITLE", "JOBS_1000"]
    ]

    if map_measure == "Jobs per 1,000":
        metric = "JOBS_1000"
        label = "Jobs per 1,000 jobs"
        prefix = ""
        suffix = ""

    elif map_measure == "Summary Statistic":
        if statistic not in valid_statistics:
            raise ValueError(
                f"Choose a Summary Statistic from: {valid_statistics}"
            )

        metric = statistic
        hourly = metric.startswith("H_")
        label = f"{metric} ($/hour)" if hourly else metric
        prefix = "$" if hourly else ""
        suffix = "/hour" if hourly else ""

    else:
        raise ValueError("Invalid map measure.")

    if occupation not in data["OCC_TITLE"].values:
        raise ValueError(f"Unknown occupational group: {occupation}")

    selected = data.loc[
        data["OCC_TITLE"] == occupation,
        ["PRIM_STATE", metric],
    ].copy()

    selected[metric] = pd.to_numeric(
        selected[metric],
        errors="coerce",
    )

    state_codes = (
        "AL AK AZ AR CA CO CT DE DC FL GA HI ID IL IN IA KS KY "
        "LA ME MD MA MI MN MS MO MT NE NV NH NJ NM NY NC ND OH "
        "OK OR PA RI SC SD TN TX UT VT VA WA WV WI WY"
    ).split()

    map_data = pd.DataFrame(
        {"PRIM_STATE": state_codes}
    ).merge(
        selected,
        on="PRIM_STATE",
        how="left",
        validate="one_to_one",
    )

    fig = px.choropleth(
        map_data,
        locations="PRIM_STATE",
        locationmode="USA-states",
        color=metric,
        scope="usa",
        color_continuous_scale="Viridis",
        labels={metric: label},
    )

    fig.update_traces(
        marker_line_color="white",
        marker_line_width=0.6,
        hovertemplate=(
            "<b>%{location}</b><br>"
            + f"{occupation}<br>"
            + f"{label}: {prefix}"
            + "%{z:,.2f}"
            + suffix
            + "<extra></extra>"
        ),
    )

    fig.update_geos(
        scope="usa",
        projection_type="albers usa",
        showland=True,
        landcolor="#E5E5E5",
        showsubunits=True,
        subunitcolor="white",
        showlakes=False,
        showframe=False,
    )

    fig.update_layout(
        title=dict(
            text=(
                f"{occupation}: {label}"
                "<br><sup>"
                "Uncolored states have no valid estimate."
                "</sup>"
            )
        ),
        template="plotly_white",
        height=600,
        margin=dict(l=10, r=10, t=100, b=20),
        coloraxis_colorbar=dict(
            title=label,
            tickformat=",.2f",
        ),
    )

    return fig
make_occupation_choropleth("Management")

The final chart is a bullet chart that compares one occupational group in a State to the national average.
It has a dropdown widget for a single occupational group State, and summary statistic. It uses the df_by_state_and_oc DataFrame.

In [13]:
def make_occupation_bullet(
    occupation,
    state,
    statistic="H_MEDIAN",
):
    """Compare a state estimate with its national occupational summary."""

    state_data = df_by_state_and_oc.copy()
    national_data = df_by_occupation.copy()

    # Summary Statistic = all non-grouped columns except JOBS_1000.
    valid_statistics = [
        column
        for column in state_data.columns
        if column != "JOBS_1000"
        and column in national_data.columns
    ]

    if statistic not in valid_statistics:
        raise ValueError(
            f"Choose a Summary Statistic from: {valid_statistics}"
        )

    # Both DataFrames retain their grouping indexes.
    state_key = (state, occupation)

    if state_key not in state_data.index:
        raise ValueError(
            f"No state summary found for {occupation} in {state}."
        )

    if occupation not in national_data.index:
        raise ValueError(
            f"No national summary found for {occupation}."
        )

    state_value = pd.to_numeric(
        state_data.loc[state_key, statistic],
        errors="coerce",
    )

    national_value = pd.to_numeric(
        national_data.loc[occupation, statistic],
        errors="coerce",
    )

    if pd.isna(state_value) or pd.isna(national_value):
        raise ValueError(
            "The selected state or national summary is missing "
            "or nonnumeric."
        )

    state_value = float(state_value)
    national_value = float(national_value)

    units = " ($/hour)" if statistic.startswith("H_") else ""
    prefix = "$" if statistic.startswith("H_") else ""

    # Leave space beyond both the state bar and reference line.
    axis_max = max(state_value, national_value, 1.0) * 1.2

    fig = go.Figure(
        go.Indicator(
            mode="number+gauge",
            value=state_value,
            number=dict(
                prefix=prefix,
                valueformat=",.2f",
            ),
            gauge=dict(
                shape="bullet",
                axis=dict(
                    range=[0, axis_max],
                    tickformat=",.2f",
                    tickprefix=prefix,
                ),
                bar=dict(
                    color="#0072B2",
                    thickness=0.5,
                ),
                bgcolor="#EEEEEE",

                # Reference line is the selected NATIONAL statistic.
                threshold=dict(
                    value=national_value,
                    line=dict(
                        color="#222222",
                        width=4,
                    ),
                    thickness=0.85,
                ),
            ),
            domain=dict(
                x=[0.05, 0.85],
                y=[0.25, 0.75],
            ),
        )
    )

    fig.update_layout(
        title=dict(
            text=(
                f"{occupation}: {state} versus national summary"
                f"<br><sup>{statistic}{units}</sup>"
            )
        ),
        template="plotly_white",
        height=350,
        margin=dict(l=50, r=50, t=100, b=80),
    )

    fig.add_annotation(
        text=(
            f"Bar — {state}: {prefix}{state_value:,.2f}"
            f" | Reference line — national summary: "
            f"{prefix}{national_value:,.2f}"
        ),
        x=0.5,
        y=0,
        xref="paper",
        yref="paper",
        showarrow=False,
    )

    return fig
make_occupation_bullet(
    occupation="Management",
    state="MI",
    statistic="H_MEDIAN",
)

# Building and exploring

Below is the code needed to generate the interactive widgets and dashboards. The dashboard is linked [here](https://mads521-dashboard.onrender.com/app). The below dashboard is simply a preview.

Step 1: Get available choices from the DataFrames, setup defaults

In [ ]:
pn.extension("plotly")

state_occupation_data = df_by_state_and_oc.reset_index()

states = sorted(
    state_occupation_data["PRIM_STATE"]
    .dropna()
    .unique()
    .tolist()
)

# Use occupations available in both DataFrames.
occupations = sorted(
    set(state_occupation_data["OCC_TITLE"].dropna())
    & set(df_by_occupation.index)
)

if "All Occupations" in occupations:
    occupations.remove("All Occupations")
    occupations.insert(0, "All Occupations")

# Summary Statistic = non-grouped columns except JOBS_1000.
# The bullet chart also requires the statistic in df_by_occupation.
summary_statistics = [
    column
    for column in df_by_state_and_oc.columns
    if column not in ["PRIM_STATE", "OCC_TITLE", "JOBS_1000"]
    and column in df_by_occupation.columns
]

if len(states) < 2:
    raise ValueError("The dashboard requires at least two states.")

if not occupations:
    raise ValueError("No occupational groups are available in both DataFrames.")

if not summary_statistics:
    raise ValueError("No shared Summary Statistic columns are available.")

default_statistic = (
    "H_MEDIAN"
    if "H_MEDIAN" in summary_statistics
    else summary_statistics[0]
)

default_occupation = (
    "All Occupations"
    if "All Occupations" in occupations
    else occupations[0]
)

default_state_1 = "MI" if "MI" in states else states[0]

default_state_2 = (
    "CA"
    if "CA" in states and default_state_1 != "CA"
    else next(state for state in states if state != default_state_1)
)

Step 2: Create shared widgets

In [19]:
summary_stat_widget = pn.widgets.Select(
    name="Summary Statistic — Charts 1, 3, and 4",
    options=summary_statistics,
    value=default_statistic,
    width=350,
)
summary_stat_widget.name = (
    "Summary Statistic — Charts 1, 3, and 4"
)
occupation_widget = pn.widgets.Select(
    name="Occupational group — map and bullet chart",
    options=occupations,
    value=default_occupation,
    width=400,
)


Step 3: Create Chart-specific widgets.

In [20]:
# Dot plot controls.
state_1_widget = pn.widgets.Select(
    name="State 1",
    options=states,
    value=default_state_1,
    width=200,
)

state_2_widget = pn.widgets.Select(
    name="State 2",
    options=states,
    value=default_state_2,
    width=200,
)

# Boxplot control: individual groups only.
box_occupations = [
    occupation
    for occupation in occupations
    if occupation != "All Occupations"
]

box_occupations_widget = pn.widgets.MultiChoice(
    name="Occupational groups — select up to 5",
    options=box_occupations,
    value=box_occupations[:3],
    sizing_mode="stretch_width",
)
# Map chart control.
map_measure_widget = pn.widgets.Select(
    name="Map measure",
    options=["Summary Statistic", "Jobs per 1,000"],
    value="Summary Statistic",
    width=250,
)

# Bullet chart control.
bullet_state_widget = pn.widgets.Select(
    name="State",
    options=states,
    value=default_state_1,
    width=200,
)

Step 4: Create rendering helper functions

In [ ]:
def plotly_pane(figure):
    return pn.pane.Plotly(
        figure,
        sizing_mode="stretch_width",
        config={
            "displaylogo": False,
            "responsive": True,
        },
    )


def render_dotplot(state_1, state_2, statistic):
    if state_1 == state_2:
        return pn.pane.Markdown(
            "**Select two different states to compare.**"
        )

    try:
        return plotly_pane(
            make_state_comparison(
                state_1,
                state_2,
                statistic,
            )
        )
    except ValueError as error:
        return pn.pane.Markdown(f"**Chart unavailable:** {error}")


def render_boxplot(selected_occupations):
    if not selected_occupations:
        return pn.pane.Markdown(
            "**Select between one and five occupational groups.**"
        )

    if len(selected_occupations) > 5:
        return pn.pane.Markdown(
            "**Select no more than five occupational groups.** "
            "Remove a selection to display the chart."
        )

    try:
        return plotly_pane(
            make_occupation_boxplot(selected_occupations)
        )
    except ValueError as error:
        return pn.pane.Markdown(f"**Chart unavailable:** {error}")


def render_map(occupation, statistic, map_measure):
    try:
        return plotly_pane(
            make_occupation_choropleth(
                occupation=occupation,
                statistic=statistic,
                map_measure=map_measure,
            )
        )
    except ValueError as error:
        return pn.pane.Markdown(
            f"**Chart unavailable:** {error}"
        )


def render_bullet(occupation, state, statistic):
    try:
        return plotly_pane(
            make_occupation_bullet(
                occupation,
                state,
                statistic,
            )
        )
    except ValueError as error:
        return pn.pane.Markdown(f"**Chart unavailable:** {error}")



Step 5: Bind charts to their respective widgets

In [21]:
dotplot_view = pn.bind(
    render_dotplot,
    state_1=state_1_widget,
    state_2=state_2_widget,
    statistic=summary_stat_widget,
)

boxplot_view = pn.bind(
    render_boxplot,
    selected_occupations=box_occupations_widget,
)

map_view = pn.bind(
    render_map,
    occupation=occupation_widget,
    statistic=summary_stat_widget,
    map_measure=map_measure_widget,
)

bullet_view = pn.bind(
    render_bullet,
    occupation=occupation_widget,
    state=bullet_state_widget,
    statistic=summary_stat_widget,
)

Step 6: Create chart sections

In [ ]:
dotplot_section = pn.Column(
    "## 1. Occupational earnings comparison",
    "Compare all occupational groups between two states.",
    pn.Row(state_1_widget, state_2_widget),
    dotplot_view,
    sizing_mode="stretch_width",
)

boxplot_section = pn.Column(
    "## 2. Occupational wage percentile summaries",
    box_occupations_widget,
    boxplot_view,
    (
        "**Box:** 25th–75th percentiles. "
        "**Middle line:** median. "
        "**Whiskers:** 10th and 90th percentiles."
    ),
    sizing_mode="stretch_width",
)

map_section = pn.Column(
    "## 3. State wage summaries and employment concentration",
    map_measure_widget,
    (
        "Use the shared occupational-group dropdown to select a group. "
        "In Summary Statistic mode, the shared Summary Statistic "
        "dropdown determines the map colors."
    ),
    map_view,
    (
        "Jobs per 1,000 measures occupational employment "
        "concentration—not job vacancies."
    ),
    sizing_mode="stretch_width",
)

bullet_section = pn.Column(
    "## 4. State versus average across states",
    bullet_state_widget,
    bullet_view,
    (
        "**Bar:** selected state's value. "
        "**Reference line:** the average across retained state "
        "estimates for the selected statistic and occupation."
    ),
    sizing_mode="stretch_width",
)


Step 7: Assemble the dashboard on one page

In [ ]:
dashboard = pn.Column(
    "# U.S. Occupational Employment and Wage Dashboard",

    "## Shared controls",
    pn.Row(
        summary_stat_widget,
        occupation_widget,
    ),
    (
    "The Summary Statistic controls Charts 1 and 4, and Chart 3 "
    "when its map measure is Summary Statistic. "
    "The shared occupational group—including All Occupations—"
    "controls Charts 3 and 4. "
    "Chart 2 has its own multiple-group selector. "
    "Wage statistics include the hourly mean, median, and percentiles."
    ),

    pn.layout.Divider(),
    dotplot_section,

    pn.layout.Divider(),
    boxplot_section,

    pn.layout.Divider(),
    map_section,

    pn.layout.Divider(),
    bullet_section,

    sizing_mode="stretch_width",
)



C:\Users\strid\AppData\Local\Programs\MUEDIT~1\Python\lib\site-packages\panel\viewable.py:298: ParamFutureWarning: Parameter 'object' on <class 'panel.pane.base.PaneBase'> is being given a valid parameter reference <function _param_bind.<locals>.wrapped at 0x000002933052B9D0> but is implicitly allow_refs=False. In future allow_refs will be enabled by default and the reference <function _param_bind.<locals>.wrapped at 0x000002933052B9D0> will be resolved to its underlying value Plotly(Figure, config={'displaylogo': False, ...}, sizing_mode='stretch_width'). Please explicitly set allow_ref on the Parameter definition to declare whether references should be resolved or not.
  super().__init__(**params)
C:\Users\strid\AppData\Local\Programs\MUEDIT~1\Python\lib\site-packages\panel\viewable.py:298: ParamFutureWarning: Parameter 'object' on <class 'panel.pane.base.PaneBase'> is being given a valid parameter reference <function _param_bind.<locals>.wrapped at 0x000002933052B670> but is implici

Here is a preview of what the actual hosted dashboard looks like (not fully functional):

In [15]:
dashboard

BokehModel(combine_events=True, render_bundle={'docs_json': {'44aa873d-749d-41b9-9518-623e9fac9736': {'version…

Questions answered:
1. What States pay the highest and lowest median wages?

        Mississippi has the lowest median wages and Massachussetts has the highest (DC ranks higher, but isn't considered a State).

2. What occupational group do the highest median earners come from?

        Management.

3. How does Michigan compare to the national median wages per occupational group?

        Michigan is a bit above average, but in the middle of the pack. No occupational group stood out as being particularly high or low.

4. Which States have the highest proportion of Computer and Mathematical occupations?

        Washington, Colorado, and Virginia have a higher proportion of Computer and Mathematical occupations.

# AI log and troubleshooting
*Note, AI was used for general questions and some code generation.
See Entries Below:



Entry 1: Library Research
- Tool: GPT-6.1
- What I asked for / what I was trying to do: I asked about what pythong plotting libraries had strong mapping functionality.
- What came back / what happened: Plotly was suggested, and after researching the docs, it met my requirements.
- What was wrong or could be better: N/A
- How I noticed: N/A
- What I did instead: N/A

Entry 2: Chart creation Dotplot
- Tool: GPT-6.1
- What I asked for / what I was trying to do: Create plotting code for the chart I described.
- What came back / what happened: It generated viable chart code, however, see the mistake it made below.
- What was wrong or could be better: It tried to "correct" my mistake of not including occupational groups, however, I didn't want to include them for that DataFrame.
- How I noticed: It directly communicated that I included an error, which wasn't the case.
- What I did instead: I reprompted, and clarified that the occupational groups should not be included.

Entry 3: Chart creation - Boxplot
- Tool: GPT-6.1
- What I asked for / what I was trying to do: Create plotting code for the chart I described.
- What came back / what happened: It generated the code, however, see the mistake it made below.
- What was wrong or could be better: I got a ValueError for the occupational groups it made.
- How I noticed: The traceback.
- What I did instead: Debugged to find out that in an earlier step, I left a hanging space, which caused none of the occ strings to be correct. It returned another error, and I used AI to debug the next bug.

Entry 4: Chart creation - Choropleth National Map
- Tool: GPT-6.1
- What I asked for / what I was trying to do: I asked if it was possible to create a national map, with state averages, and if so, how.
- What came back / what happened: It created the national Choropleth map, however, see issue below.
- What was wrong or could be better: It created multiple maps of the U.S. Also, it struggled with
- How I noticed: I saw both maps side by side.
- What I did instead: I reprompted it to only allow one occupational group per chart.

Entry 5: Chart creation - Bullet Chart
- Tool: GPT-6.1
- What I asked for / what I was trying to do: I asked if it was possible to create a national map, with state averages, and if so, how.
- What came back / what happened: It sucessfully returned code to generate a bullet chart
- What was wrong or could be better: N/A
- How I noticed: N/A
- What I did instead: N/A 

Entry 7: Create Dashboard
- Tool: GPT-6.1
- What I asked for / what I was trying to do: I asked for it to create interactable widgets for the parameters I thought would benefit from interactivity.
- What came back / what happened: It created the code, however... 
- What was wrong or could be better: I immediately got an error due to not having a module, jupyter_bokeh
- How I noticed: The traceback.
- What I did instead: I installed the module, and proceeded. It generated the dashboard very well. I made some minor modifications, and had to reprompt

Entry 8: Host on Google Colab/onrender
- Tool: GPT-6.1
- What I asked for / what I was trying to do: I asked for steps in hosting on Google Colab.
- What came back / what happened: It created the code, however... 
- What was wrong or could be better: I immediately got an error due to not having a module, jupyter_bokeh. 
- How I noticed: The traceback.
- What I did instead: After some debugging, and prompting, hosting with onrender.com seemed like the better option.

Entry 9: Include ALL occupational groups as an option
- Tool: GPT-6.1
- What I asked for / what I was trying to do: I realized I didn't include a group for ALL occupational groups, which is necessary for answering question 1.
- What came back / what happened: It created the code, however... 
- What was wrong or could be better: It made the occupational groups an option for only the map, so I had to update and make it an option as a shared dropdown object.
- How I noticed: The traceback.
- What I did instead: After some debugging, and prompting, I was able to correct it!


# Deployment instructions

*(AI generated)*

The dashboard runs as a standalone **Panel web application**. GitHub stores the application code and prepared data; Render runs the application and provides a website link. **Make sure to update the project_folder variable if you're following this process.

**An HTML export and Google Colab are not required.**

## 1. Export the prepared data

After running the notebook’s cleaning and pivot-table cells, run the following in a **code cell**:

```python
from pathlib import Path

project_folder = Path(
    r"C:\Users\strid\OneDrive\Desktop\MADS 521"
)
project_folder.mkdir(parents=True, exist_ok=True)

df_by_state_and_oc.to_csv(
    project_folder / "state_occupation_summary.csv",
    index=True,
)

df_by_occupation.to_csv(
    project_folder / "occupation_summary.csv",
    index=True,
)

for filename in [
    "state_occupation_summary.csv",
    "occupation_summary.csv",
]:
    file = project_folder / filename
    print(f"Saved: {file}")
    print(f"Exists: {file.exists()}")
```

Both files should appear in:

```text
C:\Users\strid\OneDrive\Desktop\MADS 521
```

These CSV files preserve the grouping indexes. The deployed application loads them instead of reading and cleaning the original Excel workbook.

**This Windows path is used only for exporting files locally.** The deployed application uses relative paths.

## 2. Create `app.py`

In the same folder, use a plain-text code editor to create **`app.py`**.

Make sure the filename is not `app.py.txt`. In Windows File Explorer, enable **View → Show → File name extensions** if necessary.

Start the file with:

```python
from pathlib import Path

import pandas as pd
import panel as pn
import plotly.express as px
import plotly.graph_objects as go

pn.extension("plotly")

BASE_DIR = Path(__file__).resolve().parent

df_by_state_and_oc = pd.read_csv(
    BASE_DIR / "state_occupation_summary.csv",
    index_col=["PRIM_STATE", "OCC_TITLE"],
)

df_by_occupation = pd.read_csv(
    BASE_DIR / "occupation_summary.csv",
    index_col="OCC_TITLE",
)
```

Below those lines, copy the following from the notebook, in this order:

1. The complete definition of `make_state_comparison`.
2. The complete definition of `make_occupation_boxplot`.
3. The complete definition of `make_occupation_choropleth`.
4. The complete definition of `make_occupation_bullet`.
5. The complete widget and dashboard code, beginning with:

   ```python
   state_occupation_data = df_by_state_and_oc.reset_index()
   ```

   and ending with the full assignment that creates:

   ```python
   dashboard = pn.Column(...)
   ```

For each chart function, copy everything from its `def` line through its `return fig` line. **Do not copy the preview calls that follow the function.**

At the very bottom of `app.py`, add:

```python
dashboard.servable(
    title="U.S. Occupational Employment and Wage Dashboard"
)
```

### Exclude notebook-only code

Do not include:

- `%pip install` commands.
- `import jupyter_bokeh`.
- Preview assignments and calls such as:

  ```python
  fig = make_state_comparison("MI", "CA", "H_MEDIAN")
  fig.show()
  ```

- Direct preview calls such as:

  ```python
  make_occupation_choropleth("Management").show()
  ```

- The standalone final expression `dashboard`.
- Excel loading, cleaning, or pivot-table cells.
- Colab upload or server code.
- HTML-export code.
- Markdown headings or triple-backtick code fences.

The application does not need `df_by_state` or the original Excel file for the four charts currently implemented.

## 3. Create `requirements.txt`

In the same folder, create a plain-text file named **`requirements.txt`** containing:

```text
panel
pandas
plotly
```

These are starting dependencies. After validating a working environment, record its tested package versions for reproducibility.

Your folder should contain these four deployment files, alongside any existing assignment files:

```text
C:\Users\strid\OneDrive\Desktop\MADS 521\
├── app.py
├── requirements.txt
├── state_occupation_summary.csv
└── occupation_summary.csv
```

## 4. Upload the files to GitHub

1. Sign in to [GitHub](https://github.com/).
2. Create a repository named **`mads521-dashboard`**.
3. Choose its visibility deliberately:
   - **Public:** Uploaded code and data are publicly viewable.
   - **Private:** Render needs authorization to access the repository.
4. Create the repository.
5. Select **Add file → Upload files**.
6. Upload only these four deployment files initially:
   - `app.py`
   - `requirements.txt`
   - `state_occupation_summary.csv`
   - `occupation_summary.csv`
7. Commit the upload.

Upload the files directly into the repository’s main directory, not inside an additional `MADS 521` folder.

The repository’s main page should list:

```text
app.py
requirements.txt
state_occupation_summary.csv
occupation_summary.csv
```

The analysis notebook and a `README.md` can be added for documentation.

**Uploading files to GitHub does not start the dashboard.**

## 5. Create a free Render web service

1. Sign in to [Render](https://render.com/).
2. Choose **New → Web Service**.
3. Connect GitHub and authorize access to the dashboard repository.
4. Select the **`mads521-dashboard`** repository.
5. Enter the following settings.

### Service name

```text
mads521-dashboard
```

Use another name if unavailable.

### Language

```text
Python 3
```

### Branch

Select the branch containing the uploaded files, usually:

```text
main
```

### Root directory

Leave this blank because the files are at the repository root.

### Build command

```bash
pip install -r requirements.txt
```

### Start command

```bash
python -m panel serve app.py --address 0.0.0.0 --port $PORT --allow-websocket-origin=$RENDER_EXTERNAL_HOSTNAME
```

Paste this command as written. Render supplies the port and hostname environment variables.

### Compute instance

Select **Free**. Verify that the service’s compute selection says Free; a workspace plan and a service’s compute plan are separate settings.

6. Review the configuration and any displayed charges.
7. Create the web service.

Render installs the dependencies and starts Panel. See [Render’s web-service documentation](https://render.com/docs/web-services).

## 6. Open the dashboard

Wait until Render reports that the deployment is live.

Render provides a service URL. Open that URL with **`/app`** appended:

```text
YOUR_RENDER_SERVICE_URL/app
```

Replace `YOUR_RENDER_SERVICE_URL` with the actual URL shown in Render.

The dashboard should open as a separate webpage containing all four charts and their controls. Your notebook does not need to remain open.

## 7. Test the deployed application

Before submitting the link, verify:

- State 1 and State 2 update the dot plot.
- Selecting the same state twice displays the comparison message.
- The Summary Statistic updates the dot plot and bullet chart.
- Selecting one to five occupations updates the boxplot.
- Selecting more than five occupations displays the limit message.
- The shared occupational group updates the map and bullet chart.
- The bullet chart’s state updates its bar.
- The bullet reference line matches the corresponding value in `df_by_occupation`.
- Map tooltips identify the state and jobs per 1,000 jobs.

Also check that the descriptions match the implemented charts:

- The current map shows **`JOBS_1000` only**.
- `df_by_occupation` averages retained state estimates. Label the bullet benchmark **“average across states”**, not an official national median.
- The boxplot represents averaged percentile summaries, with **10th- and 90th-percentile whiskers**.

## 8. Troubleshoot errors

Open the service’s **Logs** page in Render.

### Missing CSV file

Confirm that both CSVs are uploaded alongside `app.py`. Filenames must match exactly, including capitalization.

### Missing Python package

Confirm that `requirements.txt` contains:

```text
panel
pandas
plotly
```

### Python syntax error

Check `app.py` for copied Markdown, `%pip` commands, incomplete function definitions, or incorrect indentation.

### Dashboard page not found

Try the service URL with `/app` appended.

### Page loads but widgets do not update

Check the logs and confirm that the start command includes:

```bash
--allow-websocket-origin=$RENDER_EXTERNAL_HOSTNAME
```

### Application crashes or exceeds available memory

Inspect the logs. Confirm that the app loads the prepared CSV files and does not run the Excel processing or HTML-embedding code.

## 9. Update the application

To update chart code or dashboard controls:

1. Edit `app.py`.
2. Upload the revised file to GitHub or edit it there.
3. Commit the change.
4. If automatic deployment is enabled, Render redeploys the linked branch. Otherwise, start a manual deployment.
5. Retest the dashboard.

To update the underlying data:

1. Rerun the notebook’s data preparation.
2. Rerun the CSV-export cell.
3. Upload both updated CSVs to GitHub.
4. Commit and redeploy.

## 10. Add the live link to the notebook

After verifying the dashboard, add its actual URL to the notebook and GitHub README:

> **Live dashboard:** Add the verified Render dashboard link here.  
> The application uses Render’s free compute tier and may take time to start after a period of inactivity.

According to [Render’s free-tier documentation](https://render.com/docs/free):

- Free web services sleep after 15 minutes without incoming traffic.
- They restart when a new request arrives.
- Each workspace receives 750 free instance hours per month, shared across its free web services.
- Bandwidth and build usage have limits.
- Runtime file changes are not persistent; the CSV files deployed from GitHub remain part of the application.

**Final result:** GitHub contains the application source and prepared data, while the Render link opens the standalone interactive dashboard.

Here is the exported data. *Make sure to change your filepath if you decide to follow this process.

In [17]:
project_folder = Path(
    r"C:\Users\strid\OneDrive\Desktop\MADS 521"
)
project_folder.mkdir(parents=True, exist_ok=True)

df_by_state_and_oc.to_csv(
    project_folder / "state_occupation_summary.csv",
    index=True,
)

df_by_occupation.to_csv(
    project_folder / "occupation_summary.csv",
    index=True,
)

for filename in [
    "state_occupation_summary.csv",
    "occupation_summary.csv",
]:
    file = project_folder / filename
    print(f"Saved: {file}")
    print(f"Exists: {file.exists()}")

Saved: C:\Users\strid\OneDrive\Desktop\MADS 521\state_occupation_summary.csv
Exists: True
Saved: C:\Users\strid\OneDrive\Desktop\MADS 521\occupation_summary.csv
Exists: True
